# Analyse automatique d’un fichier PyTorch `.pt`

Ce notebook charge un fichier `.pt` ou `.pth` à partir d’un chemin fourni, puis produit une analyse structurée :

- type d’objet chargé ;
- exploration récursive des clés et sous-objets ;
- inventaire des tenseurs ;

Cette analyse est juste un test visant à comprendre la structure des données publiques servant à faire certain tests.

## 1. Paramètres



In [2]:
from pathlib import Path

# À MODIFIER : chemin vers le fichier .pt / .pth à analyser
PT_PATH = r"/NAS/coolio/Barnabe/CODES/diffusion_classifier_guidance/iguane_pt_train_dataset/aibl_127796.pt"

# Dossier de sortie pour les CSV/JSON générés par l'analyse
OUTPUT_DIR = Path("./pt_analysis_outputs")

# Sécurité :
# Laisser False si le fichier vient d'une source non totalement fiable.
# Passer à True uniquement si tu fais confiance au fichier et que le chargement sûr échoue.
TRUST_THIS_FILE = False

# Chargement sur CPU pour éviter les erreurs si le checkpoint vient d'une machine GPU
MAP_LOCATION = "cpu"

# Nombre max d'éléments utilisés pour calculer les stats d'un gros tenseur
# Les stats sur les très grands tenseurs seront échantillonnées.
MAX_ELEMENTS_FOR_STATS = 1_000_000

# Nombre max de tenseurs visualisés
N_TENSORS_TO_PLOT = 5


## 2. Imports et fonctions utilitaires

In [3]:
import os
import json
import math
import inspect
from collections import defaultdict, Counter
from typing import Any, Dict, Iterable, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import torch
except ImportError as exc:
    raise ImportError(
        "PyTorch n'est pas installé dans cet environnement. "
        "Installe-le par exemple avec : pip install torch"
    ) from exc

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PyTorch:", torch.__version__)
print("Dossier de sortie:", OUTPUT_DIR.resolve())


PyTorch: 2.6.0+cu124
Dossier de sortie: /NAS/coolio/benolive/Diffusion_beta_encoder_2D/pt_analysis_outputs


In [4]:
def human_bytes(n: float) -> str:
    if n is None or not np.isfinite(n):
        return "?"
    units = ["B", "KB", "MB", "GB", "TB"]
    n = float(n)
    for unit in units:
        if abs(n) < 1024:
            return f"{n:.2f} {unit}"
        n /= 1024
    return f"{n:.2f} PB"


def is_tensor(x: Any) -> bool:
    return isinstance(x, torch.Tensor)


def is_numpy_array(x: Any) -> bool:
    return isinstance(x, np.ndarray)


def tensor_nbytes(t: torch.Tensor) -> int:
    try:
        return int(t.numel() * t.element_size())
    except Exception:
        return 0


def short_repr(x: Any, max_len: int = 120) -> str:
    try:
        s = repr(x)
    except Exception:
        s = f"<repr impossible: {type(x).__name__}>"
    if len(s) > max_len:
        s = s[:max_len - 3] + "..."
    return s


def safe_torch_load(path: str, map_location: str = "cpu", trust_this_file: bool = False):
    """Charge un fichier PyTorch en privilégiant weights_only=True si disponible."""
    load_kwargs = {"map_location": map_location}
    sig = inspect.signature(torch.load)

    if "weights_only" in sig.parameters:
        try:
            print("Tentative de chargement sécurisé avec weights_only=True ...")
            return torch.load(path, **load_kwargs, weights_only=True)
        except Exception as exc:
            print("Le chargement sécurisé a échoué.")
            print("Erreur:", type(exc).__name__, str(exc)[:500])
            if not trust_this_file:
                raise RuntimeError(
                    "Chargement arrêté par sécurité. "
                    "Si le fichier est fiable, mets TRUST_THIS_FILE = True puis relance cette cellule."
                ) from exc

            print("TRUST_THIS_FILE=True : tentative de chargement avec weights_only=False ...")
            return torch.load(path, **load_kwargs, weights_only=False)

    print(
        "Cette version de PyTorch ne semble pas supporter weights_only. "
        "Chargement standard utilisé."
    )
    if not trust_this_file:
        print(
            "Attention : avec les anciennes versions de PyTorch, torch.load peut exécuter du code "
            "si le fichier est malveillant. Ne charge que des fichiers de confiance."
        )
    return torch.load(path, **load_kwargs)


## 3. Chargement du fichier

In [6]:
pt_path = Path(PT_PATH).expanduser().resolve()

if not pt_path.exists():
    raise FileNotFoundError(f"Fichier introuvable : {pt_path}")

if pt_path.suffix.lower() not in {".pt", ".pth", ".ckpt"}:
    print(f"Attention : extension inattendue {pt_path.suffix!r}. Analyse quand même tentée.")

print("Fichier:", pt_path)
print("Taille fichier:", human_bytes(pt_path.stat().st_size))

obj = safe_torch_load(str(pt_path), map_location=MAP_LOCATION, trust_this_file=TRUST_THIS_FILE)

print("\nObjet chargé.")
print("Type racine:", type(obj))


Fichier: /NAS/coolio/Barnabe/CODES/diffusion_classifier_guidance/iguane_pt_train_dataset/aibl_127796.pt
Taille fichier: 18.75 MB
Tentative de chargement sécurisé avec weights_only=True ...

Objet chargé.
Type racine: <class 'dict'>


## 4. Vue d’ensemble de la structure

Cette section explore récursivement l’objet chargé sans afficher tous les contenus volumineux.


In [8]:
def summarize_node(x: Any) -> Dict[str, Any]:
    row = {"type": type(x).__name__}

    if is_tensor(x):
        row.update({
            "kind": "torch.Tensor",
            "shape": tuple(x.shape),
            "dtype": str(x.dtype),
            "device": str(x.device),
            "numel": int(x.numel()),
            "memory": human_bytes(tensor_nbytes(x)),
        })
    elif is_numpy_array(x):
        row.update({
            "kind": "np.ndarray",
            "shape": tuple(x.shape),
            "dtype": str(x.dtype),
            "numel": int(x.size),
            "memory": human_bytes(x.nbytes),
        })
    elif isinstance(x, dict):
        row.update({"kind": "dict", "len": len(x), "keys_preview": list(map(str, list(x.keys())[:10]))})
    elif isinstance(x, (list, tuple)):
        row.update({"kind": type(x).__name__, "len": len(x)})
    elif isinstance(x, (str, int, float, bool, type(None))):
        row.update({"kind": "scalar/string/none", "value": short_repr(x)})
    else:
        row.update({"kind": "object", "repr": short_repr(x)})

    return row


def walk_structure(x: Any, prefix: str = "root", max_depth: int = 5, max_items_per_container: int = 50):
    rows = []
    seen = set()

    def _walk(y: Any, path: str, depth: int):
        obj_id = id(y)
        summary = summarize_node(y)
        summary["path"] = path
        summary["depth"] = depth
        rows.append(summary)

        if depth >= max_depth:
            return

        # Évite les boucles de références
        if obj_id in seen:
            return
        seen.add(obj_id)

        if isinstance(y, dict):
            for i, (k, v) in enumerate(y.items()):
                if i >= max_items_per_container:
                    rows.append({
                        "path": f"{path}.<...>",
                        "depth": depth + 1,
                        "type": "...",
                        "kind": "truncated",
                        "repr": f"{len(y) - max_items_per_container} éléments non affichés",
                    })
                    break
                _walk(v, f"{path}[{repr(k)}]", depth + 1)
        elif isinstance(y, (list, tuple)):
            for i, v in enumerate(y[:max_items_per_container]):
                _walk(v, f"{path}[{i}]", depth + 1)
            if len(y) > max_items_per_container:
                rows.append({
                    "path": f"{path}[...]",
                    "depth": depth + 1,
                    "type": "...",
                    "kind": "truncated",
                    "repr": f"{len(y) - max_items_per_container} éléments non affichés",
                })

    _walk(x, prefix, 0)
    return pd.DataFrame(rows)


structure_df = walk_structure(obj, max_depth=6, max_items_per_container=80)
pd.set_option("display.max_colwidth", 160)
display(structure_df.head(200))

structure_out = OUTPUT_DIR / "structure_summary.csv"
structure_df.to_csv(structure_out, index=False)
print("Export:", structure_out.resolve())


,type,kind,len,keys_preview,path,depth,shape,dtype,device,numel,memory,value
0,dict,dict,2.0,"[volume, ds_name]",root,0,NaN,NaN,NaN,NaN,NaN,NaN
1,Tensor,torch.Tensor,NaN,NaN,root['volume'],1,"(160, 192, 160)",torch.float32,cpu,4915200.0,18.75 MB,NaN
2,str,scalar/string/none,NaN,NaN,root['ds_name'],1,NaN,NaN,NaN,NaN,NaN,'aibl'


Export: /NAS/coolio/benolive/Diffusion_beta_encoder_2D/pt_analysis_outputs/structure_summary.csv


## 5. Inventaire des tenseurs

On récupère tous les tenseurs trouvés dans l’objet chargé, quel que soit leur emplacement.


In [9]:
def collect_tensors(x: Any, prefix: str = "root", max_depth: int = 20):
    tensors = []
    seen = set()

    def _collect(y: Any, path: str, depth: int):
        if depth > max_depth:
            return

        if is_tensor(y):
            tensors.append((path, y))
            return

        obj_id = id(y)
        if obj_id in seen:
            return
        seen.add(obj_id)

        if isinstance(y, dict):
            for k, v in y.items():
                _collect(v, f"{path}[{repr(k)}]", depth + 1)
        elif isinstance(y, (list, tuple)):
            for i, v in enumerate(y):
                _collect(v, f"{path}[{i}]", depth + 1)

    _collect(x, prefix, 0)
    return tensors


all_tensors = collect_tensors(obj)
print(f"Nombre de tenseurs trouvés : {len(all_tensors)}")

tensor_rows = []
for path, t in all_tensors:
    tensor_rows.append({
        "path": path,
        "shape": tuple(t.shape),
        "ndim": t.ndim,
        "dtype": str(t.dtype),
        "device": str(t.device),
        "requires_grad": bool(getattr(t, "requires_grad", False)),
        "numel": int(t.numel()),
        "nbytes": tensor_nbytes(t),
        "memory": human_bytes(tensor_nbytes(t)),
    })

tensor_df = pd.DataFrame(tensor_rows).sort_values("nbytes", ascending=False) if tensor_rows else pd.DataFrame()
display(tensor_df.head(100))

tensor_inventory_out = OUTPUT_DIR / "tensor_inventory.csv"
tensor_df.to_csv(tensor_inventory_out, index=False)
print("Export:", tensor_inventory_out.resolve())


Nombre de tenseurs trouvés : 1


,path,shape,ndim,dtype,device,requires_grad,numel,nbytes,memory
0,root['volume'],"(160, 192, 160)",3,torch.float32,cpu,False,4915200,19660800,18.75 MB


Export: /NAS/coolio/benolive/Diffusion_beta_encoder_2D/pt_analysis_outputs/tensor_inventory.csv
